<a href="https://colab.research.google.com/github/81Miralay81/Duzce-Anma-Video-Studio/blob/main/colab/KANATIX_Colab_Free_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KANATIX — Ücretsiz Google Colab Wan 2.2 üretim motoru (API v2)

Bu not defteri KANATIX Android uygulamasına geçici, ücretsiz bir üretim API'si sağlar.

## Kullanım
1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin.
2. **İki kod hücresini sırayla** çalıştırın. İlk kurulum model dosyalarını indirir (15-25 dk).
3. Google Drive izni isteyince onaylayın.
4. Son hücrede yazan **API ADRESİ** ve **ERİŞİM ANAHTARI**nı APK'da *Sahne Stüdyosu → Colab bağlantısı* bölümüne girip **TEST ET**'e basın. (Aynı bilgiler Drive'da `MyDrive/Kanatix/Config/colab_api.txt` dosyasında.)
5. Üretim bitene kadar Colab sekmesini açık tutun.

## v2'de ne değişti?
- Eski sürüm videoyu tek istekte bekliyordu; Cloudflare tüneli 100 saniyede cevap gelmeyen isteği **HTTP 524** ile kestiği için video hiç ulaşmıyordu. Artık **iş kuyruğu** var: APK işi gönderir, durumu sorar, bitince videoyu indirir.
- Üretim sırasında hücre çıktısının (adres/anahtar) silinmesi engellendi.
- Gelen her fotoğraf düz JPEG'e çevrilir.
- Hücre yeniden çalıştırılırsa aynı anahtar korunur, port çakışması olmaz.

## Sınırlar
- Ücretsiz Colab GPU'su garantili değildir; bir klip T4'te ~5-20 dk sürebilir.
- Colab kapanınca API de kapanır; tamamlanan MP4'ler Drive'da `MyDrive/Kanatix/ColabJobs` klasöründe kalır.

Kurulum hücresi, [Collab-AI-Gen Wan 2.2 Lightx2v notebook](https://github.com/Collab-AI-Gen/Google-Colab_Notebooks/blob/main/wan2_2/wan22_Lightx2v_Aigen.ipynb) projesinin ilk hücresini çalışma anında alır.


In [ ]:
# GPU doğrula ve upstream Wan 2.2 Lightx2v kurulumunu hazırla.
# Kurulum hatası ve eksik model/node kalırsa API hücresi çalıştırılamaz.
import requests, json, re, os, textwrap, pathlib
from IPython.display import display, Markdown

KANATIX_REQUIRED_GLOBALS = ("generate_video", "dit_model", "dit_model2", "lightx2v_lora",
                            "UnetLoaderGGUF", "WanImageToVideo", "KSamplerAdvanced")
KANATIX_MODEL_DIRS = {
    "dit_model": "/content/ComfyUI/models/diffusion_models",
    "dit_model2": "/content/ComfyUI/models/diffusion_models",
    "lightx2v_lora": "/content/ComfyUI/models/loras",
}
def _kanatix_silence_notebook_output():
    g = globals()
    g["clear_output"] = lambda *a, **k: None
    g["display_video"] = lambda *a, **k: None

def _kanatix_preflight():
    missing = [name for name in KANATIX_REQUIRED_GLOBALS if name not in globals() or globals()[name] is None]
    bad_files = []
    for name, folder in KANATIX_MODEL_DIRS.items():
        filename = globals().get(name)
        path = os.path.join(folder, str(filename)) if filename else ""
        if not path or not os.path.isfile(path) or os.path.getsize(path) < 1024 * 1024:
            bad_files.append(f"{name}: {path or 'global eksik'}")
    if missing or bad_files:
        details = []
        if missing:
            details.append("Eksik Python global/node: " + ", ".join(missing))
        if bad_files:
            details.append("Eksik/boş model dosyası: " + "; ".join(bad_files))
        raise RuntimeError("KANATIX Wan kurulumu tamamlanmadı. API başlatılmadı. " +
                           "Önce bu hücreyi yeniden çalıştırın. " + " | ".join(details))
    return True

if globals().get("_KANATIX_SETUP_COMPLETE"):
    print("Kurulum bu oturumda zaten tamamlanmış; model indirmeleri tekrarlanmayacak.")
    _kanatix_preflight()
    _kanatix_silence_notebook_output()
else:
    UPSTREAM = "https://raw.githubusercontent.com/Collab-AI-Gen/Google-Colab_Notebooks/main/wan2_2/wan22_Lightx2v_Aigen.ipynb"
    r = requests.get(UPSTREAM, timeout=60)
    r.raise_for_status()
    upstream_nb = r.json()
    setup_cell = next(c for c in upstream_nb["cells"] if c.get("cell_type") == "code")
    setup_source = "".join(setup_cell["source"])

    # RIFE bu API'de kullanılmaz; çıkarılması indirme ve ek kurulum yükünü önler.
    start = setup_source.find("%cd /content\n!git clone https://github.com/Isi-dev/Practical-RIFE")
    end = setup_source.find("%cd /content/ComfyUI\n!apt -y install -qq aria2 ffmpeg", start)
    if start >= 0 and end > start:
        setup_source = setup_source[:start] + setup_source[end:]

    setup_source = setup_source.replace("!pip install onnxruntime-gpu -y", "!pip install onnxruntime-gpu")

    # Kurulum hücresini tekrar çalıştırınca mevcut ComfyUI/node klonlarını koru.
    for command, folder in (
        ("!git clone --branch ComfyUI_v0.3.47 https://github.com/Isi-dev/ComfyUI", "/content/ComfyUI"),
        ("!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git", "/content/ComfyUI/custom_nodes/ComfyUI_GGUF"),
        ("!git clone --branch kjnv1.1.3 https://github.com/Isi-dev/ComfyUI_KJNodes.git", "/content/ComfyUI/custom_nodes/ComfyUI_KJNodes"),
    ):
        setup_source = setup_source.replace(command, f"!if [ ! -d '{folder}/.git' ]; then {command[1:]}; fi")

    # model_download mevcut ve tamamlanmış dosyayı tekrar indirmesin.
    marker = 'filename = url.split(\'/\')[-1].split(\'?\')[0]  # Remove URL parameters'
    cache_guard = marker + '\n        existing_path = os.path.join(dest_dir, filename)\n        if os.path.isfile(existing_path) and os.path.getsize(existing_path) >= 1024 * 1024:\n            print(f"Using existing model: {existing_path}")\n            return filename'
    if marker in setup_source:
        setup_source = setup_source.replace(marker, cache_guard, 1)

    # Upstream'un aria2c ile indirdiği ana text encoder/VAE/vision dosyaları için de cache kontrolü.
    setup_lines = []
    for line in setup_source.splitlines():
        if line.lstrip().startswith("!aria2c ") and "/models/" in line and " -d " in line and " -o " in line:
            url = re.search(r"(https?://\S+)", line).group(1)
            dest = re.search(r" -d ([^ ]+)", line).group(1)
            filename = re.search(r" -o ([^ ]+)", line).group(1)
            setup_lines.append(f'model_download("{url}", "{dest}", "{filename}")')
        elif any(line.lstrip().startswith(name + " = model_download(") for name in
                 ("walkingToViewersL", "walkingFromBehindL", "dancingL")):
            continue
        else:
            setup_lines.append(line)
    setup_source = "\n".join(setup_lines) + "\n"
    setup_source += "\nwalkingToViewersL = walkingFromBehindL = dancingL = None\n"

    print("GPU bilgisi:")
    !nvidia-smi
    print("Wan 2.2 upstream kurulum hücresi çalıştırılıyor; hata oluşursa burada durur.")
    result = get_ipython().run_cell(setup_source)
    if getattr(result, "error_before_exec", None) or getattr(result, "error_in_exec", None):
        error = getattr(result, "error_before_exec", None) or getattr(result, "error_in_exec", None)
        raise RuntimeError("Wan 2.2 kurulum hücresi başarısız oldu; API başlatılmadı. " + str(error)) from error
    _kanatix_preflight()
    _kanatix_silence_notebook_output()
    _KANATIX_SETUP_COMPLETE = True
    print("KANATIX Wan 2.2 kurulumu ve ön kontroller tamamlandı.")


In [ ]:
# KANATIX API v2 — iş kuyruğu (job queue) ile ücretsiz Colab oturumunu APK'ya aç
#
# NEDEN DEĞİŞTİ: Eski /generate isteği, video bitene kadar (T4'te 5-20 dk) cevap vermeden bekliyordu.
# Cloudflare tüneli 100 saniye içinde cevap gelmeyen isteği HTTP 524 ile kesiyor. Bu yüzden APK hiçbir
# zaman videoyu alamıyordu. Artık:
#   POST   /jobs            -> hemen {"job_id": ...} döner
#   GET    /jobs/<id>       -> {"status": "queued|running|done|error|cancelled", ...}
#   GET    /jobs/<id>/video -> hazır MP4
#   DELETE /jobs/<id>       -> kuyruktaki işi iptal eder
#   GET    /health          -> bağlantı testi
if not globals().get("_KANATIX_SETUP_COMPLETE"):
    raise RuntimeError("Wan 2.2 kurulum hücresi bu oturumda başarıyla tamamlanmadı. Önce üstteki kurulum hücresini çalıştırın.")
_kanatix_preflight()
_kanatix_silence_notebook_output()

import os
!pip -q install flask
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount('/content/drive')

import os, re, time, uuid, shutil, secrets, threading, subprocess, glob, hmac, json, queue, traceback
from flask import Flask, request, jsonify, send_file
from PIL import Image, ImageOps
import torch

KANATIX_DRIVE = "/content/drive/MyDrive/Kanatix/ColabJobs"
UPLOAD_DIR = "/content/kanatix_uploads"
OUTPUT_DIR = "/content/kanatix_outputs"
for _d in (KANATIX_DRIVE, UPLOAD_DIR, OUTPUT_DIR):
    os.makedirs(_d, exist_ok=True)

# Hücre tekrar çalıştırılırsa aynı anahtar, aynı kuyruk ve aynı Flask sunucusu korunur.
if "KANATIX_TOKEN" not in globals():
    KANATIX_TOKEN = secrets.token_urlsafe(32)
if "kanatix_jobs" not in globals():
    kanatix_jobs = {}
    kanatix_queue = queue.Queue()
    kanatix_jobs_lock = threading.Lock()

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU yok"
NEGATIVE = "static image, frozen frame, blurry, deformed anatomy, extra limbs, duplicate people, modern objects, fantasy, text, watermark"


def _authorized():
    return hmac.compare_digest(request.headers.get("X-Kanatix-Token", ""), KANATIX_TOKEN)


def _queue_position(job_id):
    with kanatix_jobs_lock:
        waiting = sorted((j for j in kanatix_jobs.values() if j["status"] == "queued"), key=lambda j: j["created"])
    for i, j in enumerate(waiting, 1):
        if j["id"] == job_id:
            return i
    return 0


def _run_wan(job):
    global output_path
    output_path = None
    started = time.time()
    job["stage"] = "model çalışıyor"
    generate_video(
        image_path=job["input"], positive_prompt=job["prompt"], prompt_assist="none",
        negative_prompt=NEGATIVE,
        width=480, height=832, seed=job["seed"], steps=6, cfg_scale=1.0,
        sampler_name="euler", scheduler="simple", frames=65, fps=16,
        output_format="mp4", overwrite=False,
        use_lora=False, use_lora2=False, use_lora3=False,
        use_lightx2v=True, lightx2v_Strength=3.0, lightx2v_steps=6,
        use_pusa=False, use_sage_attention=False,
        enable_flow_shift=True, shift=8.0,
        enable_flow_shift2=True, shift2=8.0, end_step1=3,
    )
    video_path = globals().get("output_path")
    if not video_path or not os.path.isfile(video_path):
        candidates = [p for p in glob.glob("/content/ComfyUI/output/*.mp4") if os.path.getmtime(p) >= started - 2]
        video_path = max(candidates, key=os.path.getmtime) if candidates else None
    if not video_path or not os.path.isfile(video_path) or os.path.getsize(video_path) < 2048:
        raise RuntimeError(f"Wan üretimi bitti ama bu işe ait MP4 bulunamadı (output_path={video_path!r}).")

    # Android'in her zaman açabileceği H.264 / yuv420p biçimine çevir.
    job["stage"] = "MP4 dönüştürülüyor"
    final_path = os.path.join(OUTPUT_DIR, f"{job['id']}.mp4")
    ffmpeg = shutil.which("ffmpeg") or "/usr/bin/ffmpeg"
    r = subprocess.run([ffmpeg, "-y", "-i", video_path, "-map", "0:v:0", "-an",
                        "-c:v", "libx264", "-preset", "veryfast", "-crf", "18",
                        "-pix_fmt", "yuv420p", "-movflags", "+faststart", final_path],
                       capture_output=True, text=True, timeout=300)
    if r.returncode != 0 or not os.path.isfile(final_path) or os.path.getsize(final_path) < 2048:
        raise RuntimeError("MP4 dönüştürme başarısız: " + (r.stderr or "ffmpeg çıktı üretmedi")[-800:])
    try:
        shutil.copy2(final_path, os.path.join(KANATIX_DRIVE, f"{job['id']}.mp4"))
    except Exception as exc:  # Drive yedeği başarısız olsa da APK videoyu alabilsin
        print("Drive yedeği alınamadı:", exc)
    return final_path


def _worker():
    while True:
        job_id = kanatix_queue.get()
        job = kanatix_jobs.get(job_id)
        if job is None or job["status"] == "cancelled":
            continue
        job["status"] = "running"
        job["started"] = time.time()
        print(f"[{time.strftime('%H:%M:%S')}] KANATIX iş başladı: {job_id}")
        try:
            job["video"] = _run_wan(job)
            job["status"] = "done"
            print(f"[{time.strftime('%H:%M:%S')}] KANATIX iş bitti: {job_id} ({int(time.time() - job['started'])} sn)")
        except torch.cuda.OutOfMemoryError:
            job["status"] = "error"
            job["error"] = "GPU belleği yetmedi (CUDA out of memory). Colab'da Çalışma zamanı > Oturumu yeniden başlat yapıp tekrar deneyin."
            traceback.print_exc()
        except Exception as exc:
            job["status"] = "error"
            job["error"] = f"{type(exc).__name__}: {str(exc)[:900]}"
            traceback.print_exc()
        finally:
            job["stage"] = ""
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass


if not globals().get("_KANATIX_WORKER_STARTED"):
    threading.Thread(target=_worker, daemon=True).start()
    _KANATIX_WORKER_STARTED = True

api = Flask("kanatix_colab_v2")
api.config["MAX_CONTENT_LENGTH"] = 25 * 1024 * 1024


@api.get("/health")
def health():
    if request.headers.get("X-Kanatix-Token") and not _authorized():
        return jsonify({"error": "Yetkisiz erişim: erişim anahtarı yanlış"}), 401
    waiting = sum(1 for j in kanatix_jobs.values() if j["status"] in ("queued", "running"))
    return jsonify({"ok": True, "api_version": 2, "service": "KANATIX Wan 2.2 Colab",
                    "gpu": GPU_NAME, "ready": bool(globals().get("_KANATIX_SETUP_COMPLETE")), "queue": waiting})


@api.post("/jobs")
def create_job():
    if not _authorized():
        return jsonify({"error": "Yetkisiz erişim: erişim anahtarı yanlış"}), 401
    image = request.files.get("image")
    if image is None:
        return jsonify({"error": "image alanı eksik"}), 400
    prompt = (request.form.get("prompt") or "").strip()
    if not prompt:
        return jsonify({"error": "prompt alanı boş"}), 400
    try:
        seed = int(request.form.get("seed", "0") or "0")
    except ValueError:
        seed = 0
    if seed <= 0:
        seed = int(time.time()) % 2147483647
    job_id = uuid.uuid4().hex
    input_path = os.path.join(UPLOAD_DIR, f"{job_id}.jpg")
    try:
        # Gelen dosya ne olursa olsun (PNG, WEBP, döndürülmüş JPEG…) düz RGB JPEG'e çevir.
        img = ImageOps.exif_transpose(Image.open(image.stream)).convert("RGB")
        img.save(input_path, "JPEG", quality=95)
    except Exception as exc:
        return jsonify({"error": f"Fotoğraf açılamadı: {exc}"}), 415
    job = {"id": job_id, "status": "queued", "stage": "", "error": "", "created": time.time(),
           "input": input_path, "prompt": prompt, "seed": seed, "video": None}
    with kanatix_jobs_lock:
        kanatix_jobs[job_id] = job
    kanatix_queue.put(job_id)
    return jsonify({"job_id": job_id, "status": "queued", "position": _queue_position(job_id)}), 202


@api.get("/jobs/<job_id>")
def job_status(job_id):
    if not _authorized():
        return jsonify({"error": "Yetkisiz erişim"}), 401
    job = kanatix_jobs.get(job_id)
    if job is None:
        return jsonify({"error": "Böyle bir iş yok (Colab yeniden başlamış olabilir)"}), 404
    elapsed = int(time.time() - job.get("started", job["created"]))
    return jsonify({"job_id": job_id, "status": job["status"], "stage": job["stage"], "error": job["error"],
                    "position": _queue_position(job_id) if job["status"] == "queued" else 0,
                    "elapsed_seconds": elapsed})


@api.get("/jobs/<job_id>/video")
def job_video(job_id):
    if not _authorized():
        return jsonify({"error": "Yetkisiz erişim"}), 401
    job = kanatix_jobs.get(job_id)
    if job is None:
        return jsonify({"error": "Böyle bir iş yok"}), 404
    if job["status"] != "done" or not job.get("video") or not os.path.isfile(job["video"]):
        return jsonify({"error": f"Video henüz hazır değil (durum: {job['status']})"}), 409
    return send_file(job["video"], mimetype="video/mp4", as_attachment=True,
                     download_name=f"kanatix-{job_id}.mp4", conditional=False)


@api.delete("/jobs/<job_id>")
def cancel_job(job_id):
    if not _authorized():
        return jsonify({"error": "Yetkisiz erişim"}), 401
    job = kanatix_jobs.get(job_id)
    if job is None:
        return jsonify({"error": "Böyle bir iş yok"}), 404
    if job["status"] == "queued":
        job["status"] = "cancelled"
    return jsonify({"job_id": job_id, "status": job["status"]})


@api.post("/generate")
def old_generate():
    return jsonify({"error": "Bu APK sürümü eski. /generate kaldırıldı; GitHub'dan son KANATIX APK'sını kurun."}), 410


# Flask sunucusu: hücre tekrar çalışırsa yeni rotalar eskisinin yerine geçer (port çakışması olmaz).
if "_kanatix_app_holder" not in globals():
    _kanatix_app_holder = {"app": api}

    def _dispatch(environ, start_response):
        return _kanatix_app_holder["app"](environ, start_response)

    def _run_api():
        from werkzeug.serving import run_simple
        run_simple("0.0.0.0", 7860, _dispatch, threaded=True, use_reloader=False)

    threading.Thread(target=_run_api, daemon=True).start()
    time.sleep(2)
else:
    _kanatix_app_holder["app"] = api
print("KANATIX API v2 çalışıyor (port 7860).")

# Hesap gerektirmeyen Cloudflare Quick Tunnel (hücre tekrar çalışırsa eski tünel kapatılır)
cloudflared = "/usr/local/bin/cloudflared"
if not os.path.exists(cloudflared):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cloudflared], check=True)
    os.chmod(cloudflared, 0o755)
if globals().get("kanatix_tunnel") is not None:
    try:
        kanatix_tunnel.terminate()
    except Exception:
        pass
log_path = "/content/kanatix-cloudflared.log"
kanatix_tunnel = subprocess.Popen([cloudflared, "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"],
                                  stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
public_url = None
for _ in range(90):
    time.sleep(2)
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError("Cloudflare ücretsiz tünel adresi alınamadı. /content/kanatix-cloudflared.log dosyasına bakın.")

# Adres ve anahtarı kendi Google Drive'ına kaydet (APK'ya girerken buradan da kopyalayabilirsin).
KANATIX_CONFIG_DIR = "/content/drive/MyDrive/Kanatix/Config"
os.makedirs(KANATIX_CONFIG_DIR, exist_ok=True)
config_data = {"service": "KANATIX Wan 2.2 Colab", "api_version": 2, "api_url": public_url,
               "token": KANATIX_TOKEN, "health_url": public_url + "/health",
               "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"), "gpu": GPU_NAME}
with open(os.path.join(KANATIX_CONFIG_DIR, "colab_api.json"), "w", encoding="utf-8") as f:
    json.dump(config_data, f, ensure_ascii=False, indent=2)
with open(os.path.join(KANATIX_CONFIG_DIR, "colab_api.txt"), "w", encoding="utf-8") as f:
    f.write(f"KANATIX API ADRESI: {public_url}\n")
    f.write(f"ERISIM ANAHTARI: {KANATIX_TOKEN}\n")
    f.write(f"GUNCELLEME: {config_data['updated_at']}\n")

print("\n" + "=" * 60)
print("KANATIX ÜCRETSİZ COLAB API v2 HAZIR")
print("GPU           :", GPU_NAME)
print("API ADRESİ    :", public_url)
print("ERİŞİM ANAHTARI:", KANATIX_TOKEN)
print("Bu iki bilgiyi APK > Sahne Stüdyosu > Colab bağlantısı'na girip TEST ET'e bas.")
print("Aynı bilgiler Drive'da: MyDrive/Kanatix/Config/colab_api.txt")
print("Bu sekmeyi üretim bitene kadar açık tut. Üretim ilerlemesi aşağıda yazacak.")
print("=" * 60)
